In [ ]:
from Bio import Entrez
import csv
import time

# Search GEO for cancer RNA-seq studies and save their linked PubMed IDs.
Entrez.email = os.environ.get("NCBI_EMAIL", "your_email@example.com")
if os.environ.get("NCBI_API_KEY"):
    Entrez.api_key = os.environ["NCBI_API_KEY"]

CANCER_TERM = "(Neoplasms[MeSH Terms]  AND gse[ETYP]) AND rnaseq counts[Filter] "


def _gds_doc_field(doc, *names):
    if not hasattr(doc, "get"):
        return None
    for name in names:
        val = doc.get(name)
        if val is not None and val != "":
            return val
    return None


def _gds_scalar_text(value):
    if value is None or value == "":
        return ""
    try:
        return str(int(value))
    except (TypeError, ValueError):
        return str(value).strip()


def _gds_pmids_cell(value):
    if value is None or value == "":
        return ""
    if isinstance(value, (list, tuple)):
        parts = [_gds_scalar_text(x) for x in value if _gds_scalar_text(x)]
        return ";".join(parts)
    return _gds_scalar_text(value)


def fetch_all_matching_gse_accessions(term, batch_size=500, output_path="all_cancer_rnaseq_gse.csv"):
    # 1) Search and store results on History server
    h = Entrez.read(
        Entrez.esearch(
            db="gds",
            term=term,
            usehistory="y",
            retmax=0,
        )
    )
    count = int(h["Count"])
    webenv = h["WebEnv"]
    query_key = h["QueryKey"]

    # 2) Page through esummary; keep gds uid, GSE accession, PubMed IDs (semicolon-separated)
    rows_by_uid = {}
    for start in range(0, count, batch_size):
        handle = Entrez.esummary(
            db="gds",
            query_key=query_key,
            WebEnv=webenv,
            retstart=start,
            retmax=batch_size,
        )
        s = Entrez.read(handle)

        if isinstance(s, dict):
            docs = s.get("DocumentSummarySet", {}).get("DocumentSummary", [])
        else:
            docs = s
        if isinstance(docs, dict):
            docs = [docs]

        for doc in docs:
            uid = _gds_doc_field(doc, "Id", "uid")
            acc = _gds_doc_field(doc, "Accession", "accession")
            pubmedids = _gds_doc_field(doc, "PubMedIds", "pubmedids")
            if uid is None or acc is None:
                continue
            acc = str(acc).strip()
            if not acc.startswith("GSE"):
                continue
            rows_by_uid[str(uid)] = {
                "gds_uid": str(uid),
                "accession": acc,
                "pmids": _gds_pmids_cell(pubmedids),
            }

        fetched_so_far = min(start + batch_size, count)
        print(f"{fetched_so_far}/{count}")

        time.sleep(1)  # be polite to NCBI; adjust if you have an API key

    ordered = sorted(rows_by_uid.values(), key=lambda r: r["accession"])
    with open(output_path, "w", encoding="utf-8", newline="") as file:
        writer = csv.DictWriter(
            file, fieldnames=["gds_uid", "accession", "pmids"]
        )
        writer.writeheader()
        writer.writerows(ordered)

    print(f"Saved {len(ordered)} rows to {output_path}")
    return output_path


fetch_all_matching_gse_accessions(CANCER_TERM)


In [ ]:
import requests
import random
import pandas as pd

PMC_IDCONV_URL = "https://pmc.ncbi.nlm.nih.gov/tools/idconv/api/v1/articles/"
PMID_PMCID_CSV = "PMID-to-PMCID.csv"
TOOL = "pubmed_retrieval"
EMAIL = os.environ.get("NCBI_EMAIL", "your_email@example.com")


df = pd.read_csv('all_cancer_rnaseq_gse.csv')

pmids = sorted(
    {
        pmid.strip()
        for value in df["pmids"].dropna().astype(str)
        for pmid in value.split(";")
        if pmid.strip()
    }
)

rows = []
for start in range(0, len(pmids), 200):
    batch = pmids[start : start + 200]
    resp = requests.get(
        PMC_IDCONV_URL,
        params={
            "ids": ",".join(batch),
            "idtype": "pmid",
            "format": "json",
            "tool": TOOL,
            "email": EMAIL,
        },
        timeout=120,
    )
    resp.raise_for_status()

    for record in resp.json().get("records", []):
        pmid = str(record.get("pmid", "")).strip()
        pmcid = str(record.get("pmcid", "")).strip()
        if pmid:
            rows.append({"pmid": pmid, "pmcid": pmcid or None})

    print(f"Mapped {min(start + 200, len(pmids))} / {len(pmids)} PMIDs")
    time.sleep(2 + 0.2 * random.random())

pmid_to_pmcid_df = pd.DataFrame(rows).drop_duplicates(subset=["pmid"])
pmid_to_pmcid_df.to_csv(PMID_PMCID_CSV, index=False)
print(f"pmid to pmcid map saved to {PMID_PMCID_CSV}")
pmid_to_pmcid_df.head()


In [ ]:
pmid_to_pmcid_lookup = {
    str(pmid): str(pmcid)
    for pmid, pmcid in pd.read_csv(PMID_PMCID_CSV)[["pmid", "pmcid"]].itertuples(index=False)
    if pd.notna(pmid) and pd.notna(pmcid) and str(pmcid).strip()
}

df_final = pd.read_csv("all_cancer_rnaseq_gse.csv")


def map_pmids_to_pmcids(pmids_value):
    if pd.isna(pmids_value):
        return None

    pmids = [pmid.strip() for pmid in str(pmids_value).split(";") if pmid.strip()]
    pmcids = [pmid_to_pmcid_lookup[pmid] for pmid in pmids if pmid in pmid_to_pmcid_lookup]

    return ";".join(pmcids) if pmcids else None


df_final["pmcids"] = df_final["pmids"].apply(map_pmids_to_pmcids)
df_final.to_csv("all_cancer_rnaseq_gse.csv", index=False)


In [ ]:
df = pd.read_csv("all_cancer_rnaseq_gse.csv")
df = df[df["pmcids"].notna() & (df["pmcids"] != "")].copy()
os.makedirs("rna_seq_data", exist_ok=True)
output_path = "rna_seq_data/rna_seq_gds_with_publication_filtered.csv"
df.to_csv(output_path, index=False)
print(f"Saved {len(df)} rows to {output_path}")
df